# MISM3: explorando as bases de dados

Este caderno lê as bases que ficam junto dele (`catalogo/`) e os arquivos de dados do site (`web/dados/`) e mostra **o que existe, o que falta e quanta confiança dá para ter em cada dado**.

**Como abrir:** na raiz do projeto, `pip install jupyter pandas` e depois `jupyter notebook catalogo/explorar_bases.ipynb`.

**Regras do projeto, valem aqui também**
- Só olhamos dados **agregados ou públicos**. Nada de CEP, endereço ou registro de paciente de pessoas.
- Todo serviço do catálogo precisa de **fonte e data de verificação**. Este caderno mostra quem está sem.
- Os dados de `web/dados/demo/` são **inventados** para demonstração. O caderno avisa quando é isso que está sendo lido.

In [ ]:
from pathlib import Path
import json
import pandas as pd

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 160)

# o caderno fica em catalogo/, a raiz do projeto é a pasta de cima
RAIZ = Path.cwd()
if not (RAIZ / "catalogo").is_dir():
    RAIZ = RAIZ.parent
CATALOGO, WEB_DADOS = RAIZ / "catalogo", RAIZ / "web" / "dados"
print("Raiz do projeto:", RAIZ)

## 1. Catálogo manual de serviços (`servicos_manuais.csv`)
Cadastro feito à mão a partir de páginas oficiais. É a base mais confiável do projeto, porque cada linha tem `fonte_url` e `verificado_em`.

In [ ]:
catalogo = pd.read_csv(CATALOGO / "servicos_manuais.csv", dtype=str).fillna("")
print(len(catalogo), "serviços cadastrados")
catalogo["tipo"].value_counts().rename("quantos").to_frame()

### Qualidade: o que cada serviço informa
Falta de telefone ou de horário não é erro: às vezes a página oficial não publica. Mas é a lista do que **ainda precisa de ligação ou conferência**.

In [ ]:
campos = ["endereco", "telefone", "horario", "cep", "fonte_url", "verificado_em"]
qualidade = pd.DataFrame({
    "tem": [(catalogo[c] != "").sum() for c in campos],
    "falta": [(catalogo[c] == "").sum() for c in campos],
}, index=campos)
qualidade["% preenchido"] = (100 * qualidade["tem"] / len(catalogo)).round(0).astype(int)
qualidade

In [ ]:
# Itens sem telefone: quem precisa de uma ligação ou de outra fonte
catalogo.loc[catalogo["telefone"] == "", ["tipo", "nome", "fonte_url"]]

In [ ]:
# Fonte e data são obrigatórias. Esta lista tem que estar VAZIA.
sem_fonte = catalogo[(catalogo["fonte_url"] == "") | (catalogo["verificado_em"] == "")]
print("Sem fonte ou data:", len(sem_fonte))
sem_fonte[["id", "nome"]]

In [ ]:
# Idade da verificação: dado velho merece conferência nova
hoje = pd.Timestamp.today().normalize()
catalogo["dias_desde_verificacao"] = (hoje - pd.to_datetime(catalogo["verificado_em"], errors="coerce")).dt.days
catalogo.sort_values("dias_desde_verificacao", ascending=False)[["nome", "verificado_em", "dias_desde_verificacao"]].head(10)

### Saúde: as UBS cadastradas
Fonte: guia oficial da Fundação Municipal de Saúde. Veja o horário e a quem cada unidade atende.

In [ ]:
ubs = catalogo[(catalogo["tipo"] == "saude") & (catalogo["subtipo"] == "ubs")]
ubs[["nome", "telefone", "horario"]].reset_index(drop=True)

In [ ]:
# Pergunta de exemplo: qual UBS atende um bairro? (a área de abrangência está na observação)
bairro = "Wenzel"
ubs[ubs["observacao"].str.contains(bairro, case=False)][["nome", "endereco", "telefone", "horario"]]

## 2. Concursos e processos seletivos (`vagas_publicas.csv`)
Só entra com o edital oficial. Vazia significa que ninguém cadastrou ainda, e não que não existem vagas.

In [ ]:
vagas = pd.read_csv(CATALOGO / "vagas_publicas.csv", dtype=str).fillna("")
print(len(vagas), "cadastros")
vagas.head()

## 3. De mulher para mulher (`mulher_para_mulher.csv`)
Só mostramos **quantos** cadastros existem. Esta base exige verificação por canal oficial, e não é exibida linha a linha aqui.

In [ ]:
m2m = pd.read_csv(CATALOGO / "mulher_para_mulher.csv", dtype=str).fillna("")
print(len(m2m), "cadastros (detalhes não exibidos)")

## 4. Dados do site (`web/dados/`)
O site usa os dados **reais** quando existem (`servicos.json`, gerado por `pipeline/montar_dados.py`) e os de **demonstração** quando não existem.

In [ ]:
real = WEB_DADOS / "servicos.json"
arquivo = real if real.is_file() else WEB_DADOS / "demo" / "servicos.json"
dados = json.loads(arquivo.read_text(encoding="utf-8"))
if arquivo == real:
    print("Lendo dados REAIS:", arquivo.name)
else:
    print("ATENÇÃO: lendo dados de DEMONSTRAÇÃO (inventados). Rode pipeline/montar_dados.py para os reais.")
servicos = pd.DataFrame(dados["servicos"])
print(len(servicos), "serviços")
servicos["tipo"].value_counts().rename("quantos").to_frame()

In [ ]:
# Quantos aparecem no mapa (têm coordenada) e quantos só na lista
com_mapa = servicos["lat"].notna().sum() if "lat" in servicos else 0
print("No mapa:", com_mapa, "| só na lista:", len(servicos) - com_mapa)
servicos["geo"].value_counts(dropna=False).rename("quantos").to_frame() if "geo" in servicos else None

## 5. Painel de gestão: os números são simulados
`web/dados/demo/gestao.json` é gerado por `pipeline/gerar_demo_gestao.py` com uma semente fixa. **Não é estatística real.**

In [ ]:
gestao = json.loads((WEB_DADOS / "demo" / "gestao.json").read_text(encoding="utf-8"))
print("Chaves:", [k for k in gestao if k != "meta"])
pd.DataFrame(gestao["necessidades"])

## 6. Saúde da mulher agregada (opcional, só se você gerou)
Só aparece se existir `dados/bruto/saude_mulher/saude_mulher_resumo.json` (gerado por `pipeline/saude_mulher.py`). Contagens menores que 5 vêm suprimidas e **a fonte ainda não foi confirmada**: não chame de incidência.

In [ ]:
resumo = RAIZ / "dados" / "bruto" / "saude_mulher" / "saude_mulher_resumo.json"
if resumo.is_file():
    r = json.loads(resumo.read_text(encoding="utf-8"))
    print("Fonte confirmada?", r["meta"]["fonte_confirmada"])
    display(pd.DataFrame(r["dados"]).head(20))
else:
    print("Resumo ainda não gerado (normal). Veja docs/integracao_saude_mulher.md.")

## 7. Para você explorar
- Quais tipos de serviço têm mais itens **sem telefone**?
- Há serviços com a mesma `fonte_url`? (`catalogo["fonte_url"].value_counts()`)
- Compare o catálogo manual com o `servicos.json` real depois de rodar o `montar_dados.py`: o que o CNES e o INEP acrescentaram?